# Pandas 파일별 보강1 · 03 Merge 다른 표의 정보를 붙이고 검증하기

**한 줄 요약** · 주문에 없는 회원 등급을 고객 파일에서 공통 Key로 붙이고, 행 수 · 연결 실패 · Key 중복을 검증한 뒤 'VIP가 더 많이 사는가'를 기준별로 답함

- **원본** · `5. pandas/01_학습노트/03_pandas_applied_merge.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/04_Pandas` 전체 정리 7장 (여러 표 연결 · how 4종 · 연결 검증 · 등급 분석)

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 필요한 정보가 어느 표에 있는지 확인하고, 두 표의 공통 Key와 Key 중복 여부를 먼저 검사한다
2. `merge(on=, how=)`의 how 4종(left · inner · right · outer)이 남기는 행을 실행 전에 말한다
3. left merge 뒤 행 수 · `isna()` · `indicator=True` · `validate`로 연결 실패와 행 증가를 확인한다
4. 등급별 건수 · 총매출 · 평균 · 1명당 매출로 'VIP가 더 많이 산다'를 기준을 밝혀 고쳐 쓴다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **이미지 경로 CSV와 라벨 CSV가 따로 있을 때** · 공통 Key로 left merge (3-1)
- **연결했더니 행 수가 늘었을 때** · Key 중복 · `validate` (3-3)
- **일부 정보가 비어 버렸을 때** · `isna()` · `indicator` (3-2 · 3-3)
- **그룹별로 '더 많이'를 답할 때** · 건수 · 합계 · 평균 · 1명당 (4장)

## 학습 지도

<img src="images/pd3_map.png" alt="Merge 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 20분

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>원본 독립 실행 준비 셀 · 원본 문제</td></tr>
<tr><td>1</td><td>정보는 어느 표에 있나</td><td>membership은 어느 파일에 있고, 두 표를 잇는 열은 무엇인가?</td></tr>
<tr><td>2</td><td>작은 표로 merge 이해</td><td>Key로 붙이면 표가 어떻게 바뀌고, how는 어느 행을 남기는가?</td></tr>
<tr><td>3</td><td>실제 연결과 검증</td><td>merge가 에러 없이 실행되면 연결에 성공한 것인가?</td></tr>
<tr><td>4</td><td>종합 실습</td><td>등급별 구매 행동을 건수 · 총매출 · 평균 · 1명당으로 비교하려면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

원본의 '독립 실행 준비' 셀을 그대로 실행해 분석용 주문 `logical_orders`(992행)를 만듦. **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · `pandas` (pandas 3 기준 실행 결과)
- **seed · 장치** · 필요 없음
- **데이터** · `data/orders.csv` 주문 1000행 · `data/customers.csv` 고객 300명
- **준비 규칙** · 할인 결측 0 · 날짜 변환 · 지역 공백 제거 · 수집 오류 제외 · 주문 ID당 한 행 (02 주문 노트북과 같음)
- **빈칸 표시** · 코드의 `# 빈칸 03-①` = 원본 노트북의 첫 번째 빈 코드 칸 (모두 9칸)

**실행 흐름**

1. `pandas` 불러오기 · 출력 설정
2. 주문 · 고객 CSV 읽기
3. 할인 전 · 후 금액 계산
4. 오류 · 중복 주문을 뺀 `logical_orders` 만들기

In [1]:
# 원본 '독립 실행 준비' 셀: 앞 노트북과 같은 기준으로 분석용 주문을 다시 만듦
# 왜 · 앞 노트북을 실행하지 않아도 같은 기준의 주문 표로 시작하기 위해
import pandas as pd  # 관례: pd 로 줄여 부름

pd.set_option("display.unicode.east_asian_width", True)  # 한글 열 맞춤
orders = pd.read_csv("data/orders.csv")                  # 주문 1000행 (raw)
customers = pd.read_csv("data/customers.csv")            # 고객 300명

# 왜 · 할인 전 · 후 금액을 한 번만 계산해 뒤 문제들이 같은 기준을 쓰게 하기 위해
orders["sales"] = orders["price"] * orders["quantity"]              # 할인 전 금액
orders["discount"] = orders["discount"].fillna(0)                   # 빈 할인율 = 할인 없음
orders["final_sales"] = orders["sales"] * (1 - orders["discount"])  # 할인 후 금액
orders["order_date"] = pd.to_datetime(orders["order_date"])         # 문자열 → 날짜
orders["region"] = orders["region"].str.strip()                     # 지역 앞뒤 공백 제거

logical_orders = (orders.loc[orders["record_status"].ne("ingestion_error")]  # 수집 오류 제외
                  .drop_duplicates(subset="order_id", keep="first").copy())  # 주문 ID당 한 행
print(pd.Series({"logical_rows": len(logical_orders),  # 행 수 = 고유 주문 ID 수
                 "unique_order_id": logical_orders["order_id"].nunique()}).to_dict())

{'logical_rows': 992, 'unique_order_id': 992}


### 원본 문제

- **질문** · VIP 고객은 정말 더 많이 구매하는가?
- **흐름** · 필요한 정보가 어디 있는지 찾기 → 공통 연결고리로 합치기 → 연결 결과 검증
- **Mission 1 ~ 3** · 주문 표의 한계 · 고객 파일 살펴보기 · 작은 표로 merge 결과 예상
- **Mission 4 · 5** · 실제 데이터 left merge · 행 수 예상 · 실행됐다고 성공인가
- **Mission 6 · Final** · membership별 건수 · 총매출 · 평균 · 연결 실패 수를 근거로 결론
- **글 빈칸** · Customer · Merge · Membership Evidence · "VIP가 더 많이 산다"를 정확하게 고치기 · 팀장에게 한 문장

**원본 힌트**

- `left_dataframe.merge(right_dataframe, on="공통열", how="left")`
- 카테고리 분석에서 쓴 `groupby + agg` 구조를 membership에 다시 사용

---
## 1. 정보는 어느 표에 있나

> **이 장의 흐름**
>
> - 0장에서 분석용 주문 992행을 준비
> - Mission 1 · 2: 주문 표의 한계 → 고객 표 확인 → 공통 Key

<img src="images/pandas_gate.png" alt="raw 1000 → verified 997 → logical 992" width="689">

**파이프라인 위치** · ① 데이터 준비 — 필요한 정보 · Key 찾기

### 1-1. `A` 주문 표의 한계 · `columns` · `nunique()` <sub>원본 Mission 1 · 빈칸 03-①</sub>

> 🎯 **주문 표만으로 회원 등급별 구매를 답할 수 있는가?**

💡 **쉽게 말하면**

- 주문 표에는 고객을 가리키는 **번호**(customer_id)만 있음.
- 등급은 다른 파일에 있음.

**실행 흐름**

1. 열 이름 목록
2. membership · customer_id 있는지
3. 주문 속 고객 수

**먼저 예측** · 주문 표에 `membership` 열이 있을까? — 답은 코드 실행 결과 아래에

In [2]:
# 빈칸 03-① 주문 데이터의 열 이름
# 왜 · 연결에 필요한 정보가 어느 표에 있는지 열 이름부터 확인하기 위해
cols = logical_orders.columns.tolist()       # 15개 열
print(len(cols), "개 열:", cols[:6], "...")  # 앞 6개만
# 필요한 정보 · Key
print("membership 있음?", "membership" in cols, "/ customer_id 있음?", "customer_id" in cols)


# 고객을 가리키는 열은 있지만 등급은 없음
# 왜 · 등급을 쓰려면 고객 표와 연결해야 한다는 이유를 확인하기 위해
print("주문 속 고객 수:", logical_orders["customer_id"].nunique())  # 서로 다른 고객 ID

15 개 열: ['order_id', 'customer_id', 'order_date', 'product_id', 'product_name', 'category'] ...
membership 있음? False / customer_id 있음? True
주문 속 고객 수: 284


**결과** · 예측 정답은 없음 — `customer_id`만 있음

- `15 개 열: ['order_id', 'customer_id', 'order_date', 'product_id', 'product_name', 'category'] ...` — 빈칸 03-① 답 (앞 6개)
- `membership 있음? False / customer_id 있음? True` — 필요한 정보는 없고 연결고리만 있음
- `주문 속 고객 수: 284` — 주문을 낸 고객은 284명

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df.columns</code></td><td>열 이름 목록</td><td>필요한 열이 있는지 먼저</td></tr>
<tr><td><code>"열" in df.columns</code></td><td>있는가 True/False</td><td>코드로 확인</td></tr>
<tr><td><code>df["id"].nunique()</code></td><td>서로 다른 ID 수</td><td>주문에 나온 고객 수</td></tr>
<tr><td><code>len(df)</code></td><td>행 수</td><td>주문 건수</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 분석에 필요한 정보가 항상 한 파일에 있지는 않음 → 질문마다 열이 있는지 먼저 확인

> ✅ **한 줄 정리** · 질문에 필요한 열이 없으면 그 정보를 가진 다른 표와 연결할 Key가 있는지부터 확인한다.

**딥러닝 코드에서는**

- 이미지 파일 목록에 라벨이 없으면 라벨 CSV를 `image_id`로 연결

### 1-2. `A` 고객 파일과 공통 Key · `head()` · `shape` · `duplicated()` <sub>원본 Mission 2 · 빈칸 03-② · ③</sub>

> 🎯 **고객 파일을 바로 연결하기 전에 무엇을 확인하는가?**

💡 **쉽게 말하면**

- 두 표에 같은 사람을 가리키는 **주민번호 같은 열**(Key)이 있어야 함.
- Key가 한 사람 한 줄이 아니면 연결할 때 행이 늘어남.

**실행 흐름**

1. 처음 3행
2. shape · columns
3. Key 중복 · 공통 열 · 등급별 고객 수

**먼저 예측** · 원본 Customer Evidence · 고객 수 · 고객 ID 열 · 등급 열은? — 답은 코드 실행 결과 아래에

In [3]:
# 빈칸 03-② 고객 파일의 처음 몇 행
print(customers[["customer_id", "customer_name", "membership"]].head(3))  # Key · 이름 · 등급


# 빈칸 03-③ 크기와 열 이름
# 왜 · 연결할 표의 크기 · 열을 먼저 확인해 결과를 예측하기 위해
print("shape:", customers.shape)               # 고객 수 · 열 수
print("columns:", customers.columns.tolist())  # 회원 등급 열 = membership


# Key 검사: 고객 ID가 한 행씩인가 · 두 표의 공통 열은?
# 왜 · Key가 중복되면 연결 뒤 행이 늘어나므로 미리 확인하기 위해
print("customer_id 중복:", customers["customer_id"].duplicated().sum())     # 0이어야 함
print("공통 열:", sorted(set(logical_orders.columns) & set(customers.columns)))  # 연결고리 후보
print("등급별 고객 수:", customers["membership"].value_counts().to_dict())  # BASIC · PLUS · VIP

  customer_id customer_name membership
0       C0001      고객 001      BASIC
1       C0002      고객 002      BASIC
2       C0003      고객 003      BASIC
shape: (300, 7)
columns: ['customer_id', 'customer_name', 'age_group', 'home_region', 'signup_date', 'membership', 'marketing_opt_in']
customer_id 중복: 0
공통 열: ['customer_id']
등급별 고객 수: {'BASIC': 180, 'PLUS': 90, 'VIP': 30}


**결과** · 예측 정답은 300명 · `customer_id` · `membership`

- `0 C0001 고객 001 BASIC` — 빈칸 03-② 답
- `shape: (300, 7)` — 빈칸 03-③ 답 · 고객 300명 · 7열
- `customer_id 중복: 0` — 한 사람 한 줄 → 연결해도 주문이 늘지 않음
- `공통 열: ['customer_id']` — 원본 '공통 연결고리' 답

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>df.head()</code> · <code>df.shape</code></td><td>모양 · 값 형식</td><td>연결 전 표 확인</td></tr>
<tr><td><code>df["key"].duplicated().sum()</code></td><td>Key 중복 수</td><td>0인지 (연결 전 필수)</td></tr>
<tr><td><code>set(a.columns) &amp; set(b.columns)</code></td><td>두 표의 같은 이름 열</td><td>Key 후보 찾기</td></tr>
<tr><td><code>value_counts()</code></td><td>등급별 고객 수</td><td>그룹 크기 미리 보기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 열 이름이 같아도 값 형식이 다르면(`C0001` vs `c0001` · 문자열 vs 정수) 연결 실패

> ✅ **한 줄 정리** · 연결 전에 고객 표의 모양 · Key 열 · Key 중복 여부를 확인한다.

**딥러닝 코드에서는**

- 메타 CSV의 `image_id`가 중복이면 같은 이미지가 두 번 학습됨

---
## 2. 작은 표로 merge 이해

> **이 장의 흐름**
>
> - 1장에서 Key가 `customer_id`임을 확인
> - Mission 3: 3행짜리 표로 결과를 먼저 예상 → how 4종 비교

<img src="images/pandas_merge.png" alt="merge: 공통 Key로 옆에 붙이기" width="808">

**파이프라인 위치** · ① 데이터 준비 — 표 연결

### 2-1. `A` 작은 표 left merge · `merge(on=, how="left")` <sub>원본 Mission 3</sub>

> 🎯 **`orders_small`에 고객 등급을 붙이면 표가 어떻게 채워지는가?**

💡 **쉽게 말하면**

- 같은 customer_id를 찾아 **옆에 정보를 붙임**.
- 고객 표의 줄 순서가 달라도 Key로 찾으므로 결과는 같음.

**실행 흐름**

1. 작은 표 두 개
2. left merge
3. 순서 섞인 고객 표로 다시

**먼저 예측** · 원본 예상 표 · C02의 membership은? — 답은 코드 실행 결과 아래에

In [4]:
# 원본 작은 표: 결과를 먼저 예상한 뒤 연결
# 왜 · merge 결과를 손으로 예측할 수 있는 작은 표로 원리를 익히기 위해
orders_small = pd.DataFrame({"customer_id": ["C01", "C02", "C03"],        # 주문 3건
                             "sales": [50000, 30000, 70000]})
customers_small = pd.DataFrame({"customer_id": ["C01", "C02", "C03"],     # 고객 3명
                                "membership": ["VIP", "BASIC", "PLUS"]})
print(orders_small.merge(customers_small, on="customer_id", how="left"))  # 같은 Key끼리 옆에


# 순서가 섞여 있어도 Key로 짝을 찾음
# 왜 · 행 순서가 아니라 Key 값으로 짝을 찾는다는 것을 확인하기 위해
shuffled = customers_small.iloc[[2, 0, 1]]  # C03 · C01 · C02 순서
# 결과는 같음
print(orders_small.merge(shuffled, on="customer_id", how="left")["membership"].tolist())

  customer_id  sales membership
0         C01  50000        VIP
1         C02  30000      BASIC
2         C03  70000       PLUS
['VIP', 'BASIC', 'PLUS']


**결과** · 예측 정답은 `BASIC`

- `0 C01 50000 VIP` — C01 → VIP
- `1 C02 30000 BASIC` — 원본 예상 표 답
- `2 C03 70000 PLUS` — C03 → PLUS
- `['VIP', 'BASIC', 'PLUS']` — 고객 표 순서가 달라도 같은 결과

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>merge(on="Key")</code></td><td>Key 값으로 짝을 찾아 옆에</td><td>다른 파일 정보 가져오기</td></tr>
<tr><td><code>concat(axis=1)</code></td><td>Key 없이 <strong>위치</strong>로 옆에</td><td>행 순서가 같을 때만</td></tr>
<tr><td><code>left_on=</code> · <code>right_on=</code></td><td>Key 열 이름이 다를 때</td><td><code>customer_id</code> ↔ <code>buyer_id</code></td></tr>
<tr><td><code>how="left"</code></td><td>왼쪽(주문) 행 전부 유지</td><td>분석 기준 표를 지킬 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 행 순서로 붙이는 `concat`을 쓰면 순서가 다를 때 엉뚱한 짝 → Key가 있으면 `merge`

> ✅ **한 줄 정리** · `merge`는 줄 순서가 아니라 Key 값으로 짝을 찾아 옆에 붙인다.

**딥러닝 코드에서는**

- 예측 표와 정답 표를 `sample_id`로 붙여 채점

### 2-2. `A` how 4종 · `how=` <sub>보충 · Merge Mental Model</sub>

> 🎯 **한쪽에만 있는 Key는 how에 따라 어떻게 처리되는가?**

💡 **쉽게 말하면**

- 두 명단을 맞대는 **4가지 규칙**.
- left = 왼쪽 전원 · inner = 양쪽 모두 · right = 오른쪽 전원 · outer = 한쪽이라도.

<img src="images/pandas_merge_how.png" alt="how 4종이 남기는 Key" width="748">

**실행 흐름**

1. C04(주문만) · C05(고객만)
2. how 4종을 차례로
3. 행 · Key · 빈 칸 비교

**먼저 예측** · `how="inner"`의 행 수는? — 답은 코드 실행 결과 아래에

In [5]:
# 주문에만 있는 C04 · 고객에만 있는 C05
# 왜 · 한쪽에만 있는 Key가 how마다 어떻게 처리되는지 보기 위한 표를 만들기 위해
left_tbl = pd.DataFrame({"customer_id": ["C01", "C02", "C04"],   # 주문 쪽
                         "sales": [50000, 30000, 90000]})
right_tbl = pd.DataFrame({"customer_id": ["C01", "C02", "C05"],  # 고객 쪽
                          "membership": ["VIP", "BASIC", "PLUS"]})


# how 4종: 어느 쪽 Key를 남기는가
# 왜 · left · inner · right · outer가 남기는 행을 나란히 비교하기 위해
for how in ["left", "inner", "right", "outer"]:               # 4가지를 차례로
    m = left_tbl.merge(right_tbl, on="customer_id", how=how)  # 이번 how로 연결
    n_na = int(m.isna().sum().sum())                          # 표 전체 빈 칸 수
    # 행 · Key · 빈 칸
    print(f"{how:5} → {len(m)}행 | Key {m['customer_id'].tolist()} | 빈 칸 {n_na}개")

left  → 3행 | Key ['C01', 'C02', 'C04'] | 빈 칸 1개
inner → 2행 | Key ['C01', 'C02'] | 빈 칸 0개
right → 3행 | Key ['C01', 'C02', 'C05'] | 빈 칸 1개
outer → 4행 | Key ['C01', 'C02', 'C04', 'C05'] | 빈 칸 2개


**결과** · 예측 정답은 2행 — 양쪽에 다 있는 C01 · C02

- `left → 3행 | Key ['C01', 'C02', 'C04'] | 빈 칸 1개` — 왼쪽 전부 · C04는 membership NaN
- `inner → 2행 | Key ['C01', 'C02'] | 빈 칸 0개` — 주문 C04가 **사라짐**
- `right → 3행 | Key ['C01', 'C02', 'C05'] | 빈 칸 1개` — 오른쪽 전부 · C05는 sales NaN
- `outer → 4행 | Key ['C01', 'C02', 'C04', 'C05'] | 빈 칸 2개` — 합집합 · 빈 칸이 양쪽에

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>how="left"</code></td><td>왼쪽 Key 전부 · 없으면 NaN</td><td>주문을 지킬 때 (기본 선택)</td></tr>
<tr><td><code>how="inner"</code></td><td>양쪽 다 있는 Key만</td><td>연결된 것만 볼 때 (행이 줄 수 있음)</td></tr>
<tr><td><code>how="right"</code></td><td>오른쪽 Key 전부</td><td>거의 안 씀 → 표 순서를 바꿔 left</td></tr>
<tr><td><code>how="outer"</code></td><td>어느 쪽이든 있으면</td><td>빠진 Key를 모두 찾을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `inner`는 연결 안 된 주문을 조용히 삭제 → 매출 합계가 줄어듦

> ✅ **한 줄 정리** · 기준 표를 지키려면 `how="left"`, 양쪽에 다 있는 것만이면 `inner`, 빠진 것을 모두 찾으려면 `outer`를 쓴다.

**딥러닝 코드에서는**

- 학습 목록(left)에 라벨 표를 붙일 때 left + 실패 확인 · inner는 샘플이 몰래 줄어듦

---
## 3. 실제 연결과 검증

> **이 장의 흐름**
>
> - 2장의 left merge를 실제 992건에 적용 (Mission 4)
> - Mission 5: 행 수 → 빈 칸 → indicator → Key 중복 순서로 검증

<img src="images/pd3_indicator.png" alt="연결 결과: both 982 · left_only 10" width="696">

**파이프라인 위치** · ① 데이터 준비 · ⑥ 평가 — 연결 검증

### 3-1. `A` left merge와 행 수 · `merge()` · `shape` <sub>원본 Mission 4 · 빈칸 03-④ · ⑤</sub>

> 🎯 **992건 주문에 고객 정보를 left merge하면 행 수는 줄어드는가, 같은가, 늘어나는가?**

💡 **쉽게 말하면**

- left는 왼쪽(주문) 줄을 모두 남김.
- 고객 Key가 한 사람 한 줄이라 늘지도 않음.

**실행 흐름**

1. left merge → merged
2. shape 전후
3. 늘어난 열

**먼저 예측** · 원본 '행 수는 줄었다 / 같다 / 늘었다' 중 무엇? — 답은 코드 실행 결과 아래에

In [6]:
# 빈칸 03-④ 주문 기준 left merge → merged
# 왜 · 주문은 하나도 잃지 않고 고객 정보만 붙이기 위해
merged = logical_orders.merge(customers, on="customer_id", how="left")  # 주문 992건 유지


# 빈칸 03-⑤ 행과 열 개수 (Merge 직후 확인)
# 왜 · 연결 뒤 행 수가 그대로인지 바로 확인해 중복 연결을 잡기 위해
print("merge 전:", logical_orders.shape, "→ 후:", merged.shape)  # 행 같음 · 열 +6
print("늘어난 열:", [c for c in merged.columns if c not in logical_orders.columns])  # 고객 열 6개

merge 전: (992, 15) → 후: (992, 21)
늘어난 열: ['customer_name', 'age_group', 'home_region', 'signup_date', 'membership', 'marketing_opt_in']


**결과** · 예측 정답은 같다 — 992행

- `merge 전: (992, 15) → 후: (992, 21)` — 빈칸 03-⑤ 답 · 행 유지 · 열 15 + 6 = 21
- `늘어난 열: ['customer_name', 'age_group', 'home_region', 'signup_date', 'membership', 'marketing_opt_in']` — 고객 표 7열 중 Key를 뺀 6열

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>행 수 같음</td><td>Key가 한 행씩 · left</td><td>정상</td></tr>
<tr><td>행 수 늘어남</td><td>오른쪽 Key 중복</td><td>중복 확인 (3-3)</td></tr>
<tr><td>행 수 줄어듦</td><td><code>inner</code> · 필터를 함께 씀</td><td>how 확인</td></tr>
<tr><td>열 수</td><td>왼쪽 + 오른쪽 - Key</td><td>Key가 한 열로 합쳐짐</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 행 수가 같아도 모든 정보가 붙은 것은 아님 → 빈 칸 확인 (3-2)

> ✅ **한 줄 정리** · left merge 직후에는 행 수가 그대로인지(`shape` 전후)부터 확인한다.

**딥러닝 코드에서는**

- 라벨을 붙인 뒤 `assert len(merged) == len(images)`

### 3-2. `A` 연결 실패 찾기 · `isna()` · `value_counts()` <sub>원본 Mission 5 · 빈칸 03-⑥ · ⑦</sub>

> 🎯 **코드가 실행됐는데 membership이 비어 있는 주문은 왜 생기는가?**

💡 **쉽게 말하면**

- 주문의 고객 ID가 고객 파일에 **없으면** 붙일 정보가 없어 NaN.
- 실행 성공과 연결 성공은 다른 말.

**실행 흐름**

1. membership NaN 개수
2. 그 주문의 customer_id
3. 고객 파일에 있는지

**먼저 예측** · membership이 빈 주문은 몇 건? — 답은 코드 실행 결과 아래에

In [7]:
# 빈칸 03-⑥ membership이 비어 있는 주문
# 왜 · 고객 정보를 못 찾은 주문이 몇 건인지 확인하기 위해
no_member = merged["membership"].isna()        # 고객 정보를 못 찾은 주문
print("membership 빈 주문:", no_member.sum())  # 실행 성공 ≠ 연결 성공


# 빈칸 03-⑦ 비어 있는 주문의 customer_id
# 왜 · 연결 실패 원인이 되는 고객 ID를 찾아내기 위해
missing_ids = merged.loc[no_member, "customer_id"].value_counts()  # ID별 주문 수
print(missing_ids.to_dict())                                       # 고객 파일에 없는 ID
print("고객 파일에 있는가:", missing_ids.index.isin(customers["customer_id"]).any())  # 없으면 False

membership 빈 주문: 10
{'C9001': 4, 'C9002': 3, 'C9003': 2, 'C9004': 1}
고객 파일에 있는가: False


**결과** · 예측 정답은 10건

- `membership 빈 주문: 10` — 빈칸 03-⑥ 답
- `{'C9001': 4, 'C9002': 3, 'C9003': 2, 'C9004': 1}` — 빈칸 03-⑦ 답 · ID 4개 · 4 + 3 + 2 + 1 = 10
- `고객 파일에 있는가: False` — 원본 '가능한 원인' 답: 고객 파일에 없는 ID (C9xxx)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>merged["열"].isna().sum()</code></td><td>연결 실패 수</td><td>빠른 확인</td></tr>
<tr><td><code>.loc[실패, "id"].value_counts()</code></td><td>어떤 ID가 몇 건</td><td>원인 찾기</td></tr>
<tr><td><code>isin(다른 표 Key)</code></td><td>고객 파일에 있는가</td><td>없는 ID인지 확인</td></tr>
<tr><td><code>indicator=True</code></td><td>줄마다 both · left_only</td><td>다음 섹션 (3-3)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 원래 값이 비어 있던 열이면 isna로는 연결 실패와 구분 불가 → `indicator` 사용

> ✅ **한 줄 정리** · merge 뒤 붙인 열의 빈 칸 수와 그 Key를 확인해 연결 실패와 원인을 찾는다.

**딥러닝 코드에서는**

- 정답 없는 샘플 ID를 목록으로 남겨 데이터 담당자에게 확인

### 3-3. `A` indicator와 validate · `indicator=True` · `validate=` <sub>원본 Mission 5 · 보충</sub>

> 🎯 **연결 성공 · 실패를 세고, 고객 표가 한 행씩인지 연결하면서 검사하려면?**

💡 **쉽게 말하면**

- `indicator`는 줄마다 **찾음 · 못 찾음 도장**을 찍음.
- `validate`는 오른쪽 Key가 중복이면 연결을 **멈춤**.

**실행 흐름**

1. indicator로 개수
2. Key 중복 표로 validate 에러
3. 검사 없이 연결하면 행 증가

**먼저 예측** · C0001을 두 번 넣은 고객 표로 연결하면 행은 몇 개? — 답은 코드 실행 결과 아래에

In [8]:
# 원본: indicator=True로 연결 결과 표시
# 왜 · 행마다 어느 쪽에서 왔는지 표시해 실패 건을 쉽게 세기 위해
merge_check = logical_orders.merge(customers, on="customer_id", how="left",  # _merge 열 추가
                                   indicator=True)
print(merge_check["_merge"].value_counts().to_dict())                        # both · left_only


# 보충: 고객 표가 한 행씩인지 연결하면서 검사
# 왜 · validate로 중복 Key를 연결 순간 에러로 잡기 위해
try:
    dup_customers = pd.concat([customers, customers.head(1)])  # C0001을 일부러 두 번
    # 검사
    logical_orders.merge(dup_customers, on="customer_id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:                              # Key 중복이면 연결을 멈춤
    print("MergeError:", str(e).split(";")[0])                 # 첫 문장만
# 몰래 늘어남
print("Key 중복이면 행:", len(logical_orders.merge(dup_customers, on="customer_id", how="left")))

{'both': 982, 'left_only': 10, 'right_only': 0}
MergeError: Merge keys are not unique in right dataset
Key 중복이면 행: 995


**결과** · 예측 정답은 995행 — C0001 주문 3건이 두 번씩

- `{'both': 982, 'left_only': 10, 'right_only': 0}` — 원본 Merge Evidence: 성공 982 · 실패 10 · 982 + 10 = 992
- `MergeError: Merge keys are not unique in right dataset` — validate가 Key 중복을 잡음
- `Key 중복이면 행: 995` — 검사 없이 연결하면 992 + 3 = 995 (조용히 늘어남)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td><code>indicator=True</code></td><td><code>_merge</code> 열: both · left_only · right_only</td><td>어느 쪽에서 실패했나</td></tr>
<tr><td><code>validate="many_to_one"</code></td><td>오른쪽 Key 중복이면 에러</td><td>주문 여러 개 : 고객 1명</td></tr>
<tr><td><code>validate="one_to_one"</code></td><td>양쪽 모두 한 행씩</td><td>1:1 정보 연결</td></tr>
<tr><td>검사 없음</td><td>중복이면 행이 몰래 늘어남</td><td>쓰지 않음</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 코드 실행 성공 ≠ 데이터 연결 성공 → 행 수 · `_merge` · `validate` 세 가지를 확인

> ✅ **한 줄 정리** · `indicator=True`로 성공 · 실패를 세고 `validate`로 Key 중복을 막아 연결을 검증한다.

**딥러닝 코드에서는**

- `merge(..., validate="many_to_one")`를 데이터 준비 코드에 기본으로 넣기

---
## 4. 종합 실습 · VIP는 정말 더 많이 사는가

> **이 장의 흐름**
>
> - 3장에서 검증한 `merged`로 Mission 6 · Final Mission
> - '더 많이 산다'를 세 질문(몇 번 · 얼마나 · 한 번에 얼마)으로 나누고 연결 실패를 따로 적음

<img src="images/pd3_membership.png" alt="등급별 건수 · 총매출 · 평균 · 1명당 매출" width="892">

**파이프라인 위치** · ⑥ 평가 — 그룹 비교 · 결론

> 🎯 **"VIP가 더 많이 산다"를 데이터가 보여주는 범위에 맞게 고치면?**

💡 **쉽게 말하면**

- '더 많이'를 건수 · 총매출 · 평균(· 1명당)으로 나눔.
- 연결 실패 10건은 숨기지 않고 따로 보고.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>등급별 count · sum · mean</td><td>빈칸 03-⑧ · groupby + agg</td></tr>
<tr><td>2</td><td>지표마다 1위</td><td><code>idxmax()</code></td></tr>
<tr><td>3</td><td>연결 실패 포함 · 고객 수로 나눈 1명당 매출</td><td>빈칸 03-⑨ · <code>dropna=False</code> · <code>nunique()</code></td></tr>
</tbody></table>

**먼저 예측** · 총매출 1위 등급은 VIP일까? — 답은 코드 실행 결과 아래에

In [9]:
# 빈칸 03-⑧ membership별 final_sales의 count · sum · mean
# 왜 · 등급마다 건수 · 총매출 · 평균을 같은 기준으로 비교하기 위해
metrics = ["count", "sum", "mean"]  # 건수 · 총매출 · 평균
membership_summary = merged.groupby("membership")["final_sales"].agg(metrics)  # 등급마다 한 줄
print(membership_summary.round(0))  # 세 등급 비교
top = membership_summary.idxmax()   # 열마다 최댓값의 등급
print("1위 → 건수:", top["count"], "/ 총매출:", top["sum"], "/ 평균:", top["mean"])  # 기준마다 1위

            count         sum      mean
membership                             
BASIC         500  25500000.0   51000.0
PLUS          332  27224000.0   82000.0
VIP           150  19926000.0  132840.0
1위 → 건수: BASIC / 총매출: PLUS / 평균: VIP


**실행 흐름**

1. 연결 실패(NaN)까지 등급별로 묶고 실패 그룹 출력
2. 등급별 주문한 고객 수 세기
3. 등급별 총매출 ÷ 고객 수 = 1명당 매출 출력

In [10]:
# 빈칸 03-⑨ Final Mission: 연결 실패까지 · 고객 1명당 매출
# 왜 · 연결 실패 그룹을 빼지 않고, 고객 수 차이를 1명당 매출로 공정하게 비교하기 위해
full = merged.groupby("membership", dropna=False)["final_sales"].agg(metrics)  # NaN 그룹 포함
print(full.round(0).tail(1))                                      # 고객 정보를 못 찾은 주문
n_buyers = merged.groupby("membership")["customer_id"].nunique()  # 등급별 주문한 고객 수
per_buyer = membership_summary["sum"] / n_buyers                  # 고객 1명당 매출
print("주문한 고객 수:", n_buyers.to_dict())                      # VIP는 29명
print("1명당 매출:", per_buyer.round(0).to_dict())                # 1명당은 VIP가 가장 큼

            count        sum      mean
membership                            
NaN            10  1350000.0  135000.0
주문한 고객 수: {'BASIC': 165, 'PLUS': 86, 'VIP': 29}
1명당 매출: {'BASIC': 154545.0, 'PLUS': 316558.0, 'VIP': 687103.0}


**결과** · 예측 정답은 아님 — 총매출 1위는 PLUS, VIP는 평균 · 1명당 1위

- `BASIC 500 25500000.0 51000.0` — Membership Evidence · 주문 건수 1위
- `PLUS 332 27224000.0 82000.0` — 총매출 1위
- `VIP 150 19926000.0 132840.0` — 평균 주문금액 1위
- `1위 → 건수: BASIC / 총매출: PLUS / 평균: VIP` — 세 질문의 답이 모두 다름
- `NaN 10 1350000.0 135000.0` — Data Quality Note · 연결 못 한 주문 10건 · 135만 원
- `주문한 고객 수: {'BASIC': 165, 'PLUS': 86, 'VIP': 29}` — VIP는 29명뿐
- `1명당 매출: {'BASIC': 154545.0, 'PLUS': 316558.0, 'VIP': 687103.0}` — 고객 1명당은 VIP가 가장 큼
- 팀장에게 한 문장 · VIP는 한 번에 가장 많이(평균 132,840원), 1명당 가장 많이 사지만 총매출은 고객이 많은 PLUS가 가장 크며, 고객 정보가 없는 주문 10건은 제외함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>등급 <code>sum</code></td><td>등급 <strong>전체</strong> 규모 (고객 수 영향)</td><td>매출 기여</td></tr>
<tr><td>등급 <code>mean</code></td><td>주문 <strong>한 건당</strong></td><td>한 번에 얼마</td></tr>
<tr><td><code>sum</code> ÷ 고객 수</td><td>고객 <strong>1명당</strong></td><td>'한 명이 더 사는가' 질문</td></tr>
<tr><td><code>groupby(dropna=False)</code></td><td>연결 실패 한 줄 포함</td><td>결론에 실패 규모 남기기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - VIP 합계가 작은 것은 고객 수(29명) 때문 → 1명당으로 비교해야 공정

> ✅ **한 줄 정리** · '더 많이 산다'는 건수 · 총매출 · 평균 · 1명당으로 나눠 확인하고, 연결 실패 건수를 함께 보고한다.

**딥러닝 코드에서는**

- 모델 평가도 지표마다 1위가 다를 수 있음 → 정확도 · F1 · 클래스별 성능을 함께 보고

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 정보 · Key 찾기</td><td>주문엔 membership 없음 · 고객 300명 · Key <code>customer_id</code> 중복 0</td><td>1-1 · 1-2</td></tr>
<tr><td>2. how 4종</td><td>left 3 · inner 2 · right 3 · outer 4행</td><td>2-1 · 2-2</td></tr>
<tr><td>3. 연결 검증</td><td>992 → 992 · 실패 10 (C9001 ~ C9004) · Key 중복이면 995행</td><td>3장</td></tr>
<tr><td>4. 등급 결론</td><td>건수 BASIC · 총매출 PLUS · 평균 · 1명당 VIP · 실패 10건 제외</td><td>4</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>Key 확인</td><td>공통 열 · <code>duplicated().sum()</code> = 0</td><td>메타 + 라벨 CSV</td></tr>
<tr><td><code>A</code></td><td>how</td><td>left 기본 · inner는 행 감소</td><td>학습 목록 보존</td></tr>
<tr><td><code>A</code></td><td>행 수 검증</td><td><code>shape</code> 전후</td><td><code>assert len(merged) == len(images)</code></td></tr>
<tr><td><code>A</code></td><td>실패 찾기</td><td><code>isna()</code> · <code>indicator</code></td><td>정답 없는 샘플</td></tr>
<tr><td><code>A</code></td><td>validate</td><td>Key 중복이면 에러</td><td>준비 코드 기본값</td></tr>
<tr><td><code>B</code></td><td>등급 결론</td><td>건수 · 합계 · 평균 · 1명당</td><td>지표별 성능 보고</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 주문 표만으로 등급별 구매를 답할 수 없는 이유는?
- **1-2** · 연결 전에 고객 표에서 확인할 것은?
- **2-1** · `merge`는 무엇으로 짝을 찾는가?
- **2-2** · `left`와 `inner`의 차이는?
- **3-1** · left merge 직후 무엇부터 확인하는가?
- **3-2** · membership이 빈 이유는?
- **3-3** · `indicator`와 `validate`는 각각 무엇을 막는가?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `merged`에서 VIP 주문만 골라 `final_sales` 최댓값 출력
2. `merge_check`에서 `left_only` 주문의 `final_sales` 합계 출력

In [11]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-2의 `right_tbl`에 C04를 추가하면 `inner`는 몇 행?
2. 3-1을 `how="inner"`로 바꾸면 행 수와 전체 `final_sales` 합계는?
3. 고객 1명당 매출 대신 고객 1명당 주문 건수로 비교하면 1위 등급은?

In [12]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · membership 열이 없고 customer_id만 있음
- **1-2** · 모양 · Key 열 · Key 중복 0
- **2-1** · 줄 순서가 아니라 Key 값
- **2-2** · 왼쪽 전부(없으면 NaN) vs 양쪽 다 있는 Key만
- **3-1** · 행 수가 그대로인가 (`shape` 전후)
- **3-2** · 고객 파일에 없는 ID (C9001 ~ C9004)
- **3-3** · 실패를 못 세는 것 · Key 중복으로 행이 늘어나는 것

#### 코드 문제 답

```python
print(merged.loc[merged["membership"] == "VIP", "final_sales"].max())  # 1032000.0
print(merge_check.loc[merge_check["_merge"] == "left_only", "final_sales"].sum())  # 1350000.0
```

#### 변형 문제 답

*문제 → 답*

- **1** · 3행 · C01 · C02 · C04
- **2** · 982행 · 74000000 - 1350000 = 72650000원
- **3** · VIP — 150 ÷ 29 = 5.17건 (PLUS 332 ÷ 86 = 3.86 · BASIC 500 ÷ 165 = 3.03)

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) 연결 뒤 행 수만 보고 '모두 연결됨'이라 씀</td><td>빈 칸을 확인하지 않음</td><td>행 수 · isna · indicator를 모두 확인</td><td>행 수 말고 실패 수도 셌나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. merge 후에 무엇을 검증하나요?**
> - 행 수가 merge 전과 같은지 확인하고, indicator=True로 left_only가 몇 건인지 셉니다.
> - 오른쪽 Key가 중복되면 행이 늘어나므로 validate="many_to_one"을 걸어 둡니다.
> - 연결 실패 건은 결론에서 숨기지 않고 따로 보고합니다.

**Q2. how="left"와 "inner"는 언제 쓰나요?**
> - 분석 기준이 되는 표를 그대로 지켜야 하면 left를 쓰고, 못 찾은 정보는 NaN으로 남겨 따로 셉니다.
> - inner는 양쪽에 모두 있는 행만 남기므로 연결 실패가 조용히 사라져 합계가 줄 수 있어, 실패 건수를 따로 적을 때만 씁니다.

**Q3. 'VIP가 더 많이 구매한다'를 어떻게 검증했나요?**
> - 더 많이를 주문 건수, 총매출, 평균 주문금액, 고객 1명당 매출로 나눠 계산했습니다.
> - VIP는 평균과 1명당 매출은 1위였지만 고객 수가 적어 총매출은 PLUS가 1위였습니다.
> - 고객 정보가 없는 주문 10건은 제외했다고 함께 적었습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>df.columns</code> · <code>in</code></td><td>DataFrame 속성 · Python 연산자</td><td>필요한 열이 있는가</td><td>1-1 <code>"membership" in cols</code></td></tr>
<tr><td><code>.nunique()</code></td><td>Series 메서드</td><td>서로 다른 값 수</td><td>1-1 <code>logical_orders["customer_id"].nunique()</code></td></tr>
<tr><td><code>.duplicated().sum()</code></td><td>Series 메서드</td><td>Key 중복 수</td><td>1-2 <code>customers["customer_id"].duplicated().sum()</code></td></tr>
<tr><td><code>.merge(on=, how=)</code></td><td>DataFrame 메서드</td><td>Key로 옆에 붙이기</td><td>2-1 <code>orders_small.merge(customers_small, ...)</code></td></tr>
<tr><td><code>how=</code> 4종</td><td>merge 인자</td><td>남길 Key 고르기</td><td>2-2 <code>left_tbl.merge(right_tbl, ..., how=how)</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>.isna()</code> · <code>.value_counts()</code></td><td>Series 메서드</td><td>연결 실패 수 · 실패 ID</td><td>3-2 <code>merged["membership"].isna()</code></td></tr>
<tr><td><code>indicator=True</code> · <code>_merge</code></td><td>merge 인자 · 결과 열</td><td>성공 · 실패 표시</td><td>3-3 <code>merge_check["_merge"].value_counts()</code></td></tr>
<tr><td><code>validate="many_to_one"</code></td><td>merge 인자</td><td>Key 중복이면 MergeError</td><td>3-3 <code>validate="many_to_one"</code></td></tr>
<tr><td><code>groupby().agg([..])</code> · <code>dropna=False</code></td><td>DataFrame · GroupBy</td><td>등급별 지표 · 실패 포함</td><td>4 <code>merged.groupby("membership", dropna=False)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>연결 전 Key 중복 · 공통 열 확인</td><td>확인 없이 바로 merge</td></tr>
<tr><td>left + 행 수 · <code>_merge</code> 확인</td><td><code>inner</code>로 실패를 조용히 삭제</td></tr>
<tr><td><code>validate</code>로 Key 중복 막기</td><td>행이 늘어난 표로 합계 계산</td></tr>
<tr><td>결론에 실패 건수 · 금액 남기기</td><td>'VIP가 더 많이 산다'처럼 기준 없는 결론</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td><code>merge</code> vs <code>concat(axis=1)</code></td><td>Key로 vs 위치로</td><td>Key가 있으면 <code>merge</code></td></tr>
<tr><td><code>left</code> vs <code>inner</code></td><td>왼쪽 유지 vs 양쪽 공통만</td><td>기본은 left</td></tr>
<tr><td><code>isna()</code> vs <code>indicator</code></td><td>빈 칸 vs 실패 표시</td><td>원래 빈 열이면 <code>indicator</code></td></tr>
<tr><td><code>validate</code> 있음 vs 없음</td><td>중복이면 에러 vs 행 증가</td><td>항상 있음</td></tr>
<tr><td>등급 <code>sum</code> vs 1명당</td><td>규모 vs 한 사람</td><td>질문에 맞게</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · how 4종 그림을 가리고 행 수 말하기</td><td>15분</td></tr>
<tr><td>7일 뒤</td><td>4장 종합 실습을 빈 셀에 직접 다시 짜기</td><td>20분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 전체 정리 7장 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 · `03_pandas_applied_merge.ipynb`의 독립 실행 준비 · Mission 1 ~ 6 · Final Mission · 글 빈칸
- 빈 코드 칸 9개는 모두 직접 풀어 실행으로 확인 · 코드의 `# 빈칸 03-①` ~ `⑨`
- how 4종(2-2) · validate(3-3) · 고객 1명당 매출(4장)은 원본 질문에 답하기 위한 보충
- 원본의 접힌 힌트(details 태그)는 0장 '원본 힌트'로 · 원본의 Colab 업로드 셀은 로컬 실행이라 뺌 · 형식 v2.1

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>독립 실행 준비 셀</td><td>주문 정리 → logical_orders 992행</td><td>0장</td></tr>
<tr><td>Mission 1 · 현재 주문 데이터의 한계</td><td>빈칸 03-①</td><td>1-1</td></tr>
<tr><td>Mission 2 · 고객 파일 · 공통 연결고리 · Key</td><td>빈칸 03-② · ③ · Customer Evidence</td><td>1-2</td></tr>
<tr><td>Mission 3 · 작은 데이터로 Merge 예상</td><td>예상 표 채우기</td><td>2-1</td></tr>
<tr><td>Merge Mental Model</td><td>주문 + 고객 + 공통 Key</td><td>2-1 · 2-2 (how 4종 보충)</td></tr>
<tr><td>Mission 4 · 실제 데이터 연결</td><td>빈칸 03-④ · ⑤ · Merge 직후 확인</td><td>3-1</td></tr>
<tr><td>Mission 5 · 실행됐다고 성공일까?</td><td>빈칸 03-⑥ · ⑦ · indicator · Merge Evidence</td><td>3-2 · 3-3</td></tr>
<tr><td>Mission 6 · Membership 분석</td><td>빈칸 03-⑧ · Membership Evidence · 문장 고치기</td><td>4 종합 실습</td></tr>
<tr><td>Applied Final Mission · 팀장에게 한 문장</td><td>빈칸 03-⑨ · Evidence 1 ~ 3 · Data Quality Note</td><td>4 종합 실습</td></tr>
<tr><td>Merge에서 배운 흐름 · 다음 단계</td><td>정보 없음 → Key → Merge → 행 수 → 실패 → 분석</td><td>그래서 오늘 무엇을 배웠는가?</td></tr>
</tbody></table>

### 전체 코드

1 ~ 4장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 주문 데이터 열 확인
2. 1-2 고객 데이터 크기 · Key 검사

In [13]:
# ---------- 1-1 ----------
# 빈칸 03-① 주문 데이터의 열 이름
# 왜 · 연결에 필요한 정보가 어느 표에 있는지 열 이름부터 확인하기 위해
cols = logical_orders.columns.tolist()       # 15개 열
print(len(cols), "개 열:", cols[:6], "...")  # 앞 6개만
# 필요한 정보 · Key
print("membership 있음?", "membership" in cols, "/ customer_id 있음?", "customer_id" in cols)


# 고객을 가리키는 열은 있지만 등급은 없음
# 왜 · 등급을 쓰려면 고객 표와 연결해야 한다는 이유를 확인하기 위해
print("주문 속 고객 수:", logical_orders["customer_id"].nunique())  # 서로 다른 고객 ID


# ---------- 1-2 ----------
# 빈칸 03-② 고객 파일의 처음 몇 행
print(customers[["customer_id", "customer_name", "membership"]].head(3))  # Key · 이름 · 등급


# 빈칸 03-③ 크기와 열 이름
# 왜 · 연결할 표의 크기 · 열을 먼저 확인해 결과를 예측하기 위해
print("shape:", customers.shape)               # 고객 수 · 열 수
print("columns:", customers.columns.tolist())  # 회원 등급 열 = membership


# Key 검사: 고객 ID가 한 행씩인가 · 두 표의 공통 열은?
# 왜 · Key가 중복되면 연결 뒤 행이 늘어나므로 미리 확인하기 위해
print("customer_id 중복:", customers["customer_id"].duplicated().sum())     # 0이어야 함
print("공통 열:", sorted(set(logical_orders.columns) & set(customers.columns)))  # 연결고리 후보
print("등급별 고객 수:", customers["membership"].value_counts().to_dict())  # BASIC · PLUS · VIP

15 개 열: ['order_id', 'customer_id', 'order_date', 'product_id', 'product_name', 'category'] ...
membership 있음? False / customer_id 있음? True
주문 속 고객 수: 284
  customer_id customer_name membership
0       C0001      고객 001      BASIC
1       C0002      고객 002      BASIC
2       C0003      고객 003      BASIC
shape: (300, 7)
columns: ['customer_id', 'customer_name', 'age_group', 'home_region', 'signup_date', 'membership', 'marketing_opt_in']
customer_id 중복: 0
공통 열: ['customer_id']
등급별 고객 수: {'BASIC': 180, 'PLUS': 90, 'VIP': 30}


**실행 흐름**

1. 2-1 작은 표로 merge 예측 · 확인
2. 2-2 how 4종 비교

In [14]:
# ---------- 2-1 ----------
# 원본 작은 표: 결과를 먼저 예상한 뒤 연결
# 왜 · merge 결과를 손으로 예측할 수 있는 작은 표로 원리를 익히기 위해
orders_small = pd.DataFrame({"customer_id": ["C01", "C02", "C03"],        # 주문 3건
                             "sales": [50000, 30000, 70000]})
customers_small = pd.DataFrame({"customer_id": ["C01", "C02", "C03"],     # 고객 3명
                                "membership": ["VIP", "BASIC", "PLUS"]})
print(orders_small.merge(customers_small, on="customer_id", how="left"))  # 같은 Key끼리 옆에


# 순서가 섞여 있어도 Key로 짝을 찾음
# 왜 · 행 순서가 아니라 Key 값으로 짝을 찾는다는 것을 확인하기 위해
shuffled = customers_small.iloc[[2, 0, 1]]  # C03 · C01 · C02 순서
# 결과는 같음
print(orders_small.merge(shuffled, on="customer_id", how="left")["membership"].tolist())


# ---------- 2-2 ----------
# 주문에만 있는 C04 · 고객에만 있는 C05
# 왜 · 한쪽에만 있는 Key가 how마다 어떻게 처리되는지 보기 위한 표를 만들기 위해
left_tbl = pd.DataFrame({"customer_id": ["C01", "C02", "C04"],   # 주문 쪽
                         "sales": [50000, 30000, 90000]})
right_tbl = pd.DataFrame({"customer_id": ["C01", "C02", "C05"],  # 고객 쪽
                          "membership": ["VIP", "BASIC", "PLUS"]})


# how 4종: 어느 쪽 Key를 남기는가
# 왜 · left · inner · right · outer가 남기는 행을 나란히 비교하기 위해
for how in ["left", "inner", "right", "outer"]:               # 4가지를 차례로
    m = left_tbl.merge(right_tbl, on="customer_id", how=how)  # 이번 how로 연결
    n_na = int(m.isna().sum().sum())                          # 표 전체 빈 칸 수
    # 행 · Key · 빈 칸
    print(f"{how:5} → {len(m)}행 | Key {m['customer_id'].tolist()} | 빈 칸 {n_na}개")

  customer_id  sales membership
0         C01  50000        VIP
1         C02  30000      BASIC
2         C03  70000       PLUS
['VIP', 'BASIC', 'PLUS']
left  → 3행 | Key ['C01', 'C02', 'C04'] | 빈 칸 1개
inner → 2행 | Key ['C01', 'C02'] | 빈 칸 0개
right → 3행 | Key ['C01', 'C02', 'C05'] | 빈 칸 1개
outer → 4행 | Key ['C01', 'C02', 'C04', 'C05'] | 빈 칸 2개


**실행 흐름**

1. 3-1 left merge · 크기 확인
2. 3-2 연결 실패 주문 찾기
3. 3-3 `indicator` · `validate`

In [15]:
# ---------- 3-1 ----------
# 빈칸 03-④ 주문 기준 left merge → merged
# 왜 · 주문은 하나도 잃지 않고 고객 정보만 붙이기 위해
merged = logical_orders.merge(customers, on="customer_id", how="left")  # 주문 992건 유지


# 빈칸 03-⑤ 행과 열 개수 (Merge 직후 확인)
# 왜 · 연결 뒤 행 수가 그대로인지 바로 확인해 중복 연결을 잡기 위해
print("merge 전:", logical_orders.shape, "→ 후:", merged.shape)  # 행 같음 · 열 +6
print("늘어난 열:", [c for c in merged.columns if c not in logical_orders.columns])  # 고객 열 6개


# ---------- 3-2 ----------
# 빈칸 03-⑥ membership이 비어 있는 주문
# 왜 · 고객 정보를 못 찾은 주문이 몇 건인지 확인하기 위해
no_member = merged["membership"].isna()        # 고객 정보를 못 찾은 주문
print("membership 빈 주문:", no_member.sum())  # 실행 성공 ≠ 연결 성공


# 빈칸 03-⑦ 비어 있는 주문의 customer_id
# 왜 · 연결 실패 원인이 되는 고객 ID를 찾아내기 위해
missing_ids = merged.loc[no_member, "customer_id"].value_counts()  # ID별 주문 수
print(missing_ids.to_dict())                                       # 고객 파일에 없는 ID
print("고객 파일에 있는가:", missing_ids.index.isin(customers["customer_id"]).any())  # 없으면 False


# ---------- 3-3 ----------
# 원본: indicator=True로 연결 결과 표시
# 왜 · 행마다 어느 쪽에서 왔는지 표시해 실패 건을 쉽게 세기 위해
merge_check = logical_orders.merge(customers, on="customer_id", how="left",  # _merge 열 추가
                                   indicator=True)
print(merge_check["_merge"].value_counts().to_dict())                        # both · left_only


# 보충: 고객 표가 한 행씩인지 연결하면서 검사
# 왜 · validate로 중복 Key를 연결 순간 에러로 잡기 위해
try:
    dup_customers = pd.concat([customers, customers.head(1)])  # C0001을 일부러 두 번
    # 검사
    logical_orders.merge(dup_customers, on="customer_id", how="left", validate="many_to_one")
except pd.errors.MergeError as e:                              # Key 중복이면 연결을 멈춤
    print("MergeError:", str(e).split(";")[0])                 # 첫 문장만
# 몰래 늘어남
print("Key 중복이면 행:", len(logical_orders.merge(dup_customers, on="customer_id", how="left")))

merge 전: (992, 15) → 후: (992, 21)
늘어난 열: ['customer_name', 'age_group', 'home_region', 'signup_date', 'membership', 'marketing_opt_in']
membership 빈 주문: 10
{'C9001': 4, 'C9002': 3, 'C9003': 2, 'C9004': 1}
고객 파일에 있는가: False
{'both': 982, 'left_only': 10, 'right_only': 0}
MergeError: Merge keys are not unique in right dataset
Key 중복이면 행: 995


**실행 흐름**

1. 등급별 세 지표
2. 연결 실패까지 · 1명당 매출

In [16]:
# ---------- 4. 종합 실습 ----------
# 빈칸 03-⑧ membership별 final_sales의 count · sum · mean
# 왜 · 등급마다 건수 · 총매출 · 평균을 같은 기준으로 비교하기 위해
metrics = ["count", "sum", "mean"]  # 건수 · 총매출 · 평균
membership_summary = merged.groupby("membership")["final_sales"].agg(metrics)  # 등급마다 한 줄
print(membership_summary.round(0))  # 세 등급 비교
top = membership_summary.idxmax()   # 열마다 최댓값의 등급
print("1위 → 건수:", top["count"], "/ 총매출:", top["sum"], "/ 평균:", top["mean"])  # 기준마다 1위


# 빈칸 03-⑨ Final Mission: 연결 실패까지 · 고객 1명당 매출
# 왜 · 연결 실패 그룹을 빼지 않고, 고객 수 차이를 1명당 매출로 공정하게 비교하기 위해
full = merged.groupby("membership", dropna=False)["final_sales"].agg(metrics)  # NaN 그룹 포함
print(full.round(0).tail(1))                                      # 고객 정보를 못 찾은 주문
n_buyers = merged.groupby("membership")["customer_id"].nunique()  # 등급별 주문한 고객 수
per_buyer = membership_summary["sum"] / n_buyers                  # 고객 1명당 매출
print("주문한 고객 수:", n_buyers.to_dict())                      # VIP는 29명
print("1명당 매출:", per_buyer.round(0).to_dict())                # 1명당은 VIP가 가장 큼

            count         sum      mean
membership                             
BASIC         500  25500000.0   51000.0
PLUS          332  27224000.0   82000.0
VIP           150  19926000.0  132840.0
1위 → 건수: BASIC / 총매출: PLUS / 평균: VIP
            count        sum      mean
membership                            
NaN            10  1350000.0  135000.0
주문한 고객 수: {'BASIC': 165, 'PLUS': 86, 'VIP': 29}
1명당 매출: {'BASIC': 154545.0, 'PLUS': 316558.0, 'VIP': 687103.0}
